# **Step 9 – Consolidación del conocimiento**
**Proyecto TA – Análisis de Datos (1INF03, 2026-2) – Equipo E10**

> *KDD, paso 9 (Fayyad, Piatetsky-Shapiro y Smyth, 1996):* «Consolidating discovered knowledge: incorporating this knowledge into another system for further actions, or documenting it and reporting it to interested parties.»

El conocimiento se consolida de tres formas:
1. **Incorporarlo a un sistema:** una función (`src/atencion.py`) que recalcula el índice cuando llegan meses nuevos de ficha, sin volver a ajustar el modelo.
2. **Entregarlo a la institución:** la lista de prioridad (`step9_lista_prioridad.csv`) y la tabla de hallazgos con sus acciones.
3. **Documentarlo:** conclusiones, recomendaciones y límites (este notebook, el informe y la presentación).

## Librerías y datos

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from src.atencion import trayectoria, puntajes, PRIORIDAD
LIM, RES = Path("../datos/2_limpios"), Path("../datos/4_resultados")
modelo = json.load(open(RES/"step7_modelo_pca.json"))
p7 = pd.read_csv(RES/"step7_puntajes.csv", sep="|")
c3, c4, c5, c6, c8 = (json.load(open(RES/f"cifras_{k}.json")) for k in ["step3", "step4", "step5", "step6", "step8"])
print("Modelo:", modelo["nombres"], "| varianza:", modelo["varianza_explicada"])

Modelo: ['Permanencia', 'Intensidad psicosocial'] | varianza: [23.77, 17.14]


## 1. Incorporar el conocimiento: recalcular el índice desde la ficha
La función `trayectoria` resume la ficha limpia (niño × mes) y `puntajes` aplica la estandarización y las cargas **guardadas** en el Step 7. Para comprobar que la función reproduce el modelo, se aplica a la ficha actual y se compara con los puntajes del Step 7. Cuando el Puericultorio tenga meses nuevos, basta con limpiar la ficha (Step 3) y llamar a estas dos funciones.

In [2]:
panel = pd.read_csv(LIM/"step3_panel.csv", sep="|")
nuevo = puntajes(trayectoria(panel), modelo)
comp = nuevo.merge(p7[["id_nna", "permanencia", "intensidad", "cuadrante"]], on="id_nna", suffixes=("", "_step7"))
dif = float(np.abs(comp[["permanencia", "intensidad"]].to_numpy() - comp[["permanencia_step7", "intensidad_step7"]].to_numpy()).max())
print(f"Niños: {len(nuevo)} | coinciden con el Step 7: {len(comp)} | diferencia máxima de puntaje: {dif:.1e} | mismo cuadrante: {(comp['cuadrante'] == comp['cuadrante_step7']).mean():.0%}")

Niños: 158 | coinciden con el Step 7: 158 | diferencia máxima de puntaje: 4.4e-16 | mismo cuadrante: 100%


## 2. Entregable para la institución: lista de prioridad
Niños del cuadrante de prioridad, ordenados por índice, con las variables que explican su ubicación y la **estabilidad** (porcentaje de remuestreos en que quedan en ese cuadrante, Step 7). Los IDs son seudónimos; la lista se revisa con el equipo del Puericultorio.

In [3]:
lista = p7[p7["cuadrante"] == PRIORIDAD].sort_values("indice_prioridad", ascending=False)
lista = lista[["id_nna", "indice_prioridad", "estabilidad_prioridad", "edad", "tiempo_hogar", "upe", "legal_definitiva", "p_sin_soporte", "p_pti_no_tiene", "p_salud_mental"]].copy()
lista["revisar_primero"] = lista["estabilidad_prioridad"] >= 80
lista.round(2).to_csv(RES/"step9_lista_prioridad.csv", sep="|", index=False)
print("Niños en la lista:", len(lista), "| para revisar primero (estabilidad ≥ 80 %):", int(lista["revisar_primero"].sum()))
lista.head(10).round(2)

Niños en la lista: 51 | para revisar primero (estabilidad ≥ 80 %): 23


,id_nna,indice_prioridad,estabilidad_prioridad,edad,tiempo_hogar,upe,legal_definitiva,p_sin_soporte,p_pti_no_tiene,p_salud_mental,revisar_primero
0,PPA_FN_0078_M_20110808,4.10,85.67,14.98,170.40,Lima Este,1,1.00,0.89,0.0,True
1,PPA_FN_0072_M_20080622,4.00,85.00,18.11,150.43,Lima Norte-Callao,1,1.00,1.00,0.0,True
2,PPA_FN_0029_F_20081203,3.92,54.00,17.66,187.25,Lima Norte-Callao,1,0.89,1.00,0.0,False
3,PPA_FN_0039_F_20120610,3.84,83.67,14.14,150.43,Lima Norte-Callao,1,0.95,1.00,0.0,True
4,PPA_FN_0043_F_20130904,3.56,87.33,12.90,116.69,Lima Este,1,1.00,0.89,0.0,True
5,PPA_FN_0028_F_20150131,3.55,87.33,11.50,115.51,Lima,1,1.00,0.89,0.0,True
6,PPA_FN_0064_M_20130128,3.55,87.33,13.50,115.51,Lima,1,1.00,0.89,0.0,True
7,PPA_FN_0032_F_20121005,3.35,78.67,13.82,149.34,Lima,1,0.79,0.89,0.0,False
8,PPA_FN_0065_M_20161004,3.33,56.33,9.82,115.51,Otra,1,1.00,0.89,0.0,False
9,PPA_FN_0027_F_20100926,3.33,56.33,15.84,115.51,Lima,1,1.00,0.89,0.0,False


## 3. Hallazgos y acciones

In [4]:
hall = pd.DataFrame([
    {"hallazgo": "La atención psicosocial no aumenta con la permanencia (dos dimensiones independientes)",
     "evidencia": f"PCA: 2 dimensiones reales y estables (Tucker {min(c6['tucker']):.2f}–{max(c6['tucker']):.2f}); {c6['pca_varianza']} % de la varianza",
     "acción sugerida": "Revisar el plan de atención de los niños de larga permanencia"},
    {"hallazgo": f"{c8['cuadrantes'][PRIORIDAD]} niños de larga permanencia con muy poca atención",
     "evidencia": f"{c8['prioridad_pct_sin_soporte']} % sin soporte la mayoría de meses; {c8['prioridad_pct_sin_pti']} % sin PTI; 0 % con salud mental especializada",
     "acción sugerida": "Revisar la lista caso por caso, empezando por los 23 más estables (≥ 80 % de remuestreos)"},
    {"hallazgo": "La permanencia difiere entre UPE; la atención no depende de la UPE ni de la edad",
     "evidencia": f"Kruskal-Wallis p < 0,001 (permanencia) y p = {c8['p_intensidad_upe']:.2f} (intensidad)",
     "acción sugerida": "Revisar con cada UPE los tiempos de permanencia y la elaboración del PTI"},
    {"hallazgo": "El exceso de peso es generalizado y no depende de la atención",
     "evidencia": f"z de IMC medio +{c4['z_imc_medio']}; ≈ 40 % con exceso en todos los hogares, edades y cuadrantes",
     "acción sugerida": "Intervención nutricional institucional (alimentación, actividad física), no focalizada"},
    {"hallazgo": "La ficha no refleja bien el estado nutricional",
     "evidencia": f"Concordancia ficha–OMS {c5['concordancia']} %; la etiqueta no cambió en 19 meses; cortes más laxos que la OMS",
     "acción sugerida": "Calcular los z OMS de forma automática (src/oms.py) y actualizar la ficha con cada medición"},
    {"hallazgo": "La ficha no registra variables que permitan perfiles de niños",
     "evidencia": "Todo clustering repite la situación legal o el estado predominante",
     "acción sugerida": "Registrar motivo de ingreso detallado, desempeño escolar y conducta para análisis futuros"}])
hall.to_csv(RES/"step9_hallazgos.csv", sep="|", index=False); hall

,hallazgo,evidencia,acción sugerida
0,La atención psicosocial no aumenta con la perm...,PCA: 2 dimensiones reales y estables (Tucker 0...,Revisar el plan de atención de los niños de la...
1,51 niños de larga permanencia con muy poca ate...,84.3 % sin soporte la mayoría de meses; 62.7 %...,"Revisar la lista caso por caso, empezando por ..."
2,La permanencia difiere entre UPE; la atención ...,"Kruskal-Wallis p < 0,001 (permanencia) y p = 0...",Revisar con cada UPE los tiempos de permanenci...
3,El exceso de peso es generalizado y no depende...,z de IMC medio +0.98; ≈ 40 % con exceso en tod...,Intervención nutricional institucional (alimen...
4,La ficha no refleja bien el estado nutricional,Concordancia ficha–OMS 68.8 %; la etiqueta no ...,Calcular los z OMS de forma automática (src/om...
5,La ficha no registra variables que permitan pe...,Todo clustering repite la situación legal o el...,"Registrar motivo de ingreso detallado, desempe..."


## 4. Conclusiones
1. **La pregunta del proyecto cambió con la evidencia.** Clasificar la etiqueta nutricional (avance) daba AUC 0,85, pero imitaba un registro manual con rezago. El estado nutricional no forma grupos, y los grupos de la ficha ya estaban escritos en ella. La reducción de dimensiones fue el único método que encontró estructura real, estable y no trivial.
2. **Conocimiento nuevo:** la atención registrada se organiza en **permanencia** e **intensidad psicosocial**, y ambas son **independientes**. Eso deja a un tercio de los niños, los de larga permanencia, con poca atención.
3. **Conocimiento útil:** una lista de 51 niños para revisar (23 estables, para revisar primero) y una función para actualizarla con cada mes de ficha.
4. **Hallazgo complementario:** el exceso de peso (≈ 40 %) es un problema de toda la institución.

**Límites:** 158 niños de una sola institución; la ficha registra la atención que se da, no las necesidades del niño (un niño sin soporte registrado puede no necesitarlo o no tenerlo anotado); la segunda dimensión es algo menos estable (Tucker p10 = 0,71); el período cubre 19 meses.

**Trabajo futuro:** validar la lista y los nombres de las dimensiones con el equipo del Puericultorio; recalcular el índice cada trimestre; incorporar variables clínicas y escolares a la ficha.

## Verificación y cifras de este notebook

In [5]:
assert dif < 1e-9 and len(comp) == len(p7) and (comp["cuadrante"] == comp["cuadrante_step7"]).all()
assert len(lista) == 51
cifras = {"diferencia_max_funcion": dif, "ninos_lista": len(lista), "revisar_primero": int(lista["revisar_primero"].sum())}
json.dump(cifras, open(RES/"cifras_step9.json", "w"), indent=1); cifras

{'diferencia_max_funcion': 4.440892098500626e-16,
 'ninos_lista': 51,
 'revisar_primero': 23}